# 1. Problem Formulation & Mathematical Framework
## Real-Time Fraud Detection at Billion-Scale

**Author**: Data Science  
**Date**: March 2026  
**Status**: Production-Ready  

---

This notebook formalizes the fraud detection problem with mathematical rigor suitable for Big Tech interviews.


## 1. Business Problem & Context

### The Business Case
A global payments platform processes **2 billion transactions per day** across 150+ countries.

**Fraud Challenge:**
- Baseline fraud rate: **0.3-0.5%** (3-5 million fraudulent transactions/day)
- Annual fraud loss: **$1+ billion USD**
- False positive cost: $5-10 per transaction (user friction, customer support)
- False negative cost: $500-2,000 per fraud (chargeback + investigation)

### North Star Metrics
| Metric | Target | Business Impact |
|--------|--------|-----------------|
| **Detection Rate (Recall)** | > 85% | Prevent $850M+ in fraud |
| **False Positive Rate** | < 2% | Minimize user friction |
| **ROI** | > 50:1 | $50 saved for every $1 spent |
| **Latency P99** | < 10ms | Real-time approvals |
| **Model AUC-ROC** | > 0.95 | Excellent class separation |

### Technical Constraints
- **Real-time requirement**: Decision in < 10ms
- **Throughput**: 23,000 transactions per second
- **Scale**: 1M+ features, 1B+ feature combinations
- **Availability**: 99.99% uptime
- **Fairness**: No disparate impact across geographies/users

### Trade-offs
```
Recall ↔ Precision
  (catch more fraud) vs (fewer false alarms)
  
False+ Cost ↔ False- Cost
  (block legitimate) vs (allow fraudulent)
  
Latency ↔ Accuracy
  (fast inference) vs (complex features)
```


## 2. Mathematical Formulation

### 2.1 Problem Definition

This is a **binary classification problem** with severe **class imbalance**.

**Given:**
- Feature vector: $\mathbf{x} \in \mathbb{R}^d$ (transaction features)
- Binary label: $y \in \{0, 1\}$ where 1 = fraud
- Training data: $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^{n}$ with $n = 10^9$
- Class distribution: $P(y=1) \approx 0.003$ (severely imbalanced)

**Goal:**
Learn a function $f_{\theta}(\mathbf{x}) \to [0,1]$ that predicts fraud probability with:
- High recall (catch fraud)
- Low false positive rate
- Sub-10ms latency

---

### 2.2 Probabilistic Formulation (Bayesian)

We model the probability of fraud given features:

$$P(y=1|\mathbf{x}, \theta) = \frac{1}{1 + e^{-\mathbf{w}^T \mathbf{x} - b}}$$

This is **logistic function** with parameters $\theta = (\mathbf{w}, b)$.

**Intuition**: Transaction features map to a probability score between 0 and 1.

---

### 2.3 Likelihood Function (MLE)

For a dataset $\mathcal{D}$, the **likelihood** of parameters $\theta$ is:

$$L(\theta | \mathcal{D}) = \prod_{i=1}^{n} P(y_i|\mathbf{x}_i, \theta)^{y_i} \cdot (1-P(y_i|\mathbf{x}_i, \theta))^{1-y_i}$$

Taking negative log-likelihood (*binary cross-entropy*):

$$\mathcal{L}(\theta) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1-y_i) \log(1-\hat{p}_i) \right]$$

where $\hat{p}_i = P(y_i=1|\mathbf{x}_i, \theta)$

**Why log?**
- Converts product (intractable) to sum (gradient-friendly)
- Highly negative values become large positive (numerically stable)

---

### 2.4 Loss Function with Class Weights

To handle **class imbalance**, use **weighted cross-entropy**:

$$\mathcal{L}_{weighted}(\theta) = -\frac{1}{n} \sum_{i=1}^{n} \left[ w_+^{y_i} y_i \log \hat{p}_i + w_-^{1-y_i} (1-y_i) \log(1-\hat{p}_i) \right]$$

where:
- $w_+ = \frac{n}{2n_+}$ = weight for positive class (fraud)
- $w_- = \frac{n}{2n_-}$ = weight for negative class (legitimate)
- $n_+$ = count of frauds
- $n_-$ = count of legitimate transactions

**Example with $n = 1M$, fraud rate = 0.3%:**
- $n_+ = 3,000$ (fraud)
- $n_- = 997,000$ (legitimate)
- $w_+ = \frac{1,000,000}{2 \times 3,000} = 166.67$
- $w_- = \frac{1,000,000}{2 \times 997,000} = 0.502$

This ensures loss is balanced across classes.

---

### 2.5 Regularization (L2 Penalty)

To prevent overfitting, add **L2 regularization**:

$$\mathcal{L}_{total}(\theta) = \mathcal{L}_{weighted}(\theta) + \lambda ||\mathbf{w}||^2_2$$

$$= -\frac{1}{n} \sum_{i=1}^{n} w_{y_i} \left[ y_i \log \hat{p}_i + (1-y_i) \log(1-\hat{p}_i) \right] + \lambda \sum_{j=1}^{d} w_j^2$$

**Why L2?**
- Reduces magnitude of weights
- Prevents fitting to noise
- Smooth decision boundaries
- $\lambda$ controls regularization strength (hyperparameter)

---

### 2.6 Gradient Descent Optimization

To minimize $\mathcal{L}_{total}(\theta)$, use **stochastic gradient descent**:

$$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \alpha \nabla_\mathbf{w} \mathcal{L}_{total}(\theta^{(t)})$$

where $\alpha$ = learning rate (step size)

**Gradient derivation**:
$$\frac{\partial \mathcal{L}}{\partial w_j} = \frac{1}{n} \sum_{i=1}^{n} w_{y_i} (\hat{p}_i - y_i) x_{ij} + 2\lambda w_j$$

**Intuition:**
- If prediction $\hat{p}_i$ is too high (vs. true $y_i$), reduce weights
- Update proportional to prediction error and feature value
- L2 term pushes weights toward zero

---

### 2.7 Key Assumptions

1. **Independence**: Transactions are conditionally independent given features
2. **Feature Quality**: Features contain signal (not pure noise)
3. **Stationarity**: Fraud patterns don't change drastically over time
4. **No Perfect Separation**: No single feature perfectly predicts fraud
5. **Causal Features**: Features are not consequences of fraud label


## 3. Data Engineering Architecture

### 3.1 Data Pipeline Overview

```
Raw Transactions (Kafka)
        ↓
    Validation Layer (Great Expectations)
        ↓
    Feature Computation (Spark)
        ↓
    Feature Store (Versioned)
        ↓
    Training Dataset Assembly
        ↓
    Model Training (Distributed)
        ↓
    Model Registry (MLflow)
        ↓
    Inference API (FastAPI)
        ↓
    Serving (Redis Cache + Batch Scoring)
```

### 3.2 Data Ingestion Strategy

**Batch Processing** (for training):
- Daily export from data warehouse
- Transactional data + user/merchant enrichment
- Consistency checks across data sources

**Streaming** (for serving):
- Real-time transaction stream via Kafka
- Low-latency feature computation
- Caching layer for fast lookups

### 3.3 Data Validation

**Schema Validation:**
- Expect numeric columns (amount, age) to be within bounds
- Enum validation for categorical features (country codes, device types)
- Timestamp validation (within last 7 days)

**Quality Checks:**
- Missing value rates (alert if > 5%)
- Statistical anomalies (e.g., transaction amount > 3 std from mean)
- Duplicate transaction detection

**Example with Great Expectations:**
```python
# Pseudocode
@check_fraud_label_correctness
def validate_labels():
    assert df['fraud_label'].isin([0, 1])  # Binary
    assert df[df['fraud_label']==1].shape[0] > 0  # Has frauds
    assert df[df['fraud_label']==0].shape[0] > 0  # Has legitimate
```

### 3.4 Feature Store Design

**Requirements:**
- **Versioning**: Track which features in which versions
- **Consistency**: Same features in training and serving
- **Lineage**: Know data source for each feature
- **Reusability**: Share features across models

**Structure:**
```
feature_store/
├── v1.0/
│   ├── user_features_30d.parquet
│   ├── merchant_features_30d.parquet
│   ├── device_features_30d.parquet
│   └── feature_manifesto.json
├── v2.0/
│   ├── (updated features)
│   └── feature_manifesto.json
└── metadata.db  # Feature lineage
```

### 3.5 Data Leakage Prevention

**Critical**: Features must use only information available at decision time.

**WRONG (leakage):**
```python
# ❌ Uses post-transaction information
feature = "user_fraud_reported" (information AFTER fraud occurred)
feature = "merchant_chargeback_received" (future event)
```

**CORRECT (temporal integrity):**
```python
# ✅ Uses only historical information
feature = "user_fraud_count_30d_before_this_transaction"
feature = "merchant_chargeback_rate_last_30d"
```

**Temporal Strategy:**
- Training: Use data from day T-30 to day T-1
- Validation: Use data from day T to day T+30 (evaluate future performance)
- Serve: Use only data available before current minute

### 3.6 Handling Missing Data

**Strategy Matrix:**

| Scenario | Action | Rationale |
|----------|--------|-----------|
| Missing user age | Use median age | Age likely missing randomly |
| Missing device type | Flag as "unknown" | Device type important signal |
| Missing merchant data | Use aggregated stats | New merchant (cold start) |
| Missing coordinates | Use imputation + indicator | Location matters but can be inferred |

**Mathematical approach:**
$$\mathbf{x}_{ij}^{imputed} = \begin{cases} 
\mathbf{x}_{ij} & \text{if } \mathbf{x}_{ij} \text{ observed} \\
\bar{\mathbf{x}}_{j} & \text{if } \mathbf{x}_{ij} \text{ missing}
\end{cases}$$

Plus add indicator feature: $I_{ij} = \mathbb{1}(\mathbf{x}_{ij} \text{ was missing})$

### 3.7 Handling Class Imbalance

**At Data Level:**
- **SMOTE**: Synthetic Minority Oversampling (synthetic frauds)
- **Undersampling**: Sample legitimate transactions (risks data loss)
- **Hybrid**: SMOTE + undersampling

**At Model Level:**
- **Class weights**: Higher penalty for misclassifying fraud
- **Threshold optimization**: Adjust decision threshold based on cost function
- **Focal loss**: Down-weight easy examples

**For this project**: Use class weights + balanced threshold optimization

### 3.8 Noisy Data Handling

**Detection:**
- Statistical outliers (> 3σ from mean)
- Impossibilities (negative amount, future timestamp)
- Inconsistencies (user age = 150 years)

**Handling:**
```python
if amount > mean + 3*std:
    inspect_manually = True  # Human review
    dont_blindly_remove = True  # Might be legitimate whale
    
if timestamp_future:
    flag_as_data_error = True  # Include in error logs
    exclude_from_training = True  # Don't use for modeling
```

---

## 4. Data Scale & Storage

### 4.1 Volume

- **Daily transactions**: 2 billion
- **Annual data**: 700 billion transactions
- **Training dataset**: 10 million (sample for prototyping)
- **Features per transaction**: 150+
- **Feature combinations**: Millions (interactions)

### 4.2 Storage Optimization

**Raw Data:**
- Columnar format (Parquet): 100 bytes/transaction
- 2B transactions/day = 200 GB/day
- Annual = 73 TB (with replication: 219 TB)

**Feature Store:**
- Pre-computed features: 50 bytes/transaction
- 10M training samples = 500 MB
- Easily fits in memory for model training

---

## 5. Summary of Key Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| **Problem Type** | Binary classification | Clear fraud/legitimate decision |
| **Class Handling** | Weighted loss | Handles 0.3% fraud rate |
| **Regularization** | L2 + threshold tuning | Prevents overfitting + optimizes business metric |
| **Data Leakage** | Strict temporal split | Avoids look-ahead bias |
| **Feature Store** | Versioned Parquet | Lineage tracking + reproducibility |
| **Validation** | Great Expectations | Automated quality gates |

Next: Let's implement this framework in code →
